# NCAA Data Analysis

**Design Thinking Question:** Does 3-point shooting affect win rate in NCAA basketball?

**Dataset:** `data/combined_ncaa.csv` — produced by `data_preparation.ipynb`

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

DATA_DIR = os.path.join(os.getcwd(), 'data')
df = pd.read_csv(os.path.join(DATA_DIR, 'combined_ncaa.csv'))

print(f'Shape  : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Years  : {df["YEAR"].min()} – {df["YEAR"].max()}')
print(f'Columns: {list(df.columns)}')

## Column Reference

All columns in the combined dataset, grouped by category.

### Identity

| Column | Type | Description |
|---|---|---|
| `YEAR` | int | Season year |
| `TEAM ID` | int | Unique numeric team identifier |
| `TEAM` | str | Team name |
| `CONF` | str | Conference (e.g. ACC, SEC, B12, MAC) |

### Season Record

| Column | Type | Description |
|---|---|---|
| `W` | int | Total wins (regular season + tournament) |
| `L` | int | Total losses |
| `win_pct` | float | Win percentage — W / (W + L) |
| `GAMES` | int | Total games played |

### Tournament Outcome

| Column | Type | Description |
|---|---|---|
| `SEED` | float | NCAA tournament seed (1–16); NaN if team did not qualify |
| `ROUND` | int | Raw round encoding — bracket size at exit: 0=No Tourney, 68=First Four, 64=R64, 32=R32, 16=S16, 8=E8, 4=F4, 2=Final, 1=Champion |
| `tournament_round` | int | Ordinal mapping of ROUND — 0 (no tourney) to 8 (champion); higher = deeper run |
| `made_tournament` | int | Binary — 1 if team reached the 64-team bracket or better, else 0 |
| `reached_s16` | int | Binary — 1 if team reached at least the Sweet 16, else 0 |

### Efficiency Metrics

| Column | Type | Description |
|---|---|---|
| `KADJ EM` | float | **KenPom Adjusted Efficiency Margin** — net points per 100 possessions vs average D1 opponent. The headline KenPom rating. Higher = better overall team |
| `KADJ O` | float | **KenPom Adjusted Offensive Efficiency** — points scored per 100 possessions, adjusted for opponent defensive quality |
| `KADJ D` | float | **KenPom Adjusted Defensive Efficiency** — points allowed per 100 possessions, adjusted for opponent offensive quality. Lower = better defence |
| `BARTHAG` | float | **Barttorvik win probability** vs an average D1 team (0–1 scale) |
| `BADJ EM` | float | **Barttorvik Adjusted Efficiency Margin** — Barttorvik's version of net efficiency per 100 possessions |
| `PPPO` | float | **Points per possession (offence)** — raw unadjusted scoring efficiency |

### 3-Point Shooting — Offence & Defence

| Column | Type | Description |
|---|---|---|
| `3PT%` | float | Offensive 3-point field goal percentage |
| `3PTR` | float | **3PT attempt rate** — % of all offensive FG attempts that are 3-pointers. Measures how 3PT-heavy the offence is |
| `3PT%D` | float | **Defensive 3PT FG% allowed** — % of opponent 3-point attempts that go in |
| `3PTRD` | float | **Opponent 3PT attempt rate** — % of opponent FG attempts that are 3-pointers |
| `THREES FG%` | float | Offensive 3PT FG% from Shooting Splits (mirrors `3PT%`) |
| `THREES SHARE` | float | Share of offensive shots that are 3-pointers — from Shooting Splits (mirrors `3PTR`) |
| `THREES FG%D` | float | Defensive 3PT FG% allowed — from Shooting Splits |
| `THREES D SHARE` | float | Share of opponent shots that are 3-pointers — from Shooting Splits |

### 2-Point Shooting — Offence & Defence

| Column | Type | Description |
|---|---|---|
| `2PT%` | float | Offensive 2-point field goal percentage |
| `2PT%D` | float | Defensive 2-point field goal percentage allowed |
| `2PTR` | float | 2PT attempt rate — share of offensive shots that are 2-pointers |
| `2PTRD` | float | Opponent 2PT attempt rate |
| `CLOSE TWOS FG%` | float | FG% on close 2-pointers — layups and short paint shots (offence) |
| `CLOSE TWOS SHARE` | float | Share of offensive shots that are close 2-pointers (%) |
| `FARTHER TWOS FG%` | float | FG% on mid-range / farther 2-pointers (offence) |
| `FARTHER TWOS SHARE` | float | Share of offensive shots that are farther 2-pointers (%) |
| `DUNKS FG%` | float | FG% on dunks — typically near 100% |
| `DUNKS SHARE` | float | Share of offensive shots that are dunks (%) |

### Effective Shooting (Combined)

| Column | Type | Description |
|---|---|---|
| `EFG%` | float | **Effective FG% (offence)** — adjusts for 3-pointers being worth more: (FGM + 0.5 x 3PM) / FGA |
| `EFG%D` | float | **Effective FG% (defence)** — opponent EFG% allowed |

### Pace & Free Throws

| Column | Type | Description |
|---|---|---|
| `KADJ T` | float | **KenPom Adjusted Tempo** — possessions per 40 minutes, adjusted for opponent pace |
| `K TEMPO` | float | **Raw tempo** — actual possessions per game |
| `FT%` | float | Free throw percentage |
| `FTR` | float | **Free throw rate** — FT attempts per FG attempt. Higher = team draws more fouls |

### Team Context

| Column | Type | Description |
|---|---|---|
| `TALENT` | float | Composite talent rating based on recruiting rankings |
| `EXP` | float | Team experience — weighted average years in program for players |
| `WAB` | float | **Wins Above Bubble** — wins vs what a bubble-level team would earn on the same schedule |
| `ELITE SOS` | float | **Strength of schedule** — quality of opponents faced |

### Engineered Features

Derived during data preparation — not present in the raw source files.

| Column | Type | Description |
|---|---|---|
| `three_rate_category` | category | Quartile bucket of `3PTR` — Low (Q1) / Mid-Low (Q2) / Mid-High (Q3) / High (Q4) |
| `three_pct_category` | category | Quartile bucket of `3PT%` — Low (Q1) / Mid-Low (Q2) / Mid-High (Q3) / High (Q4) |
| `three_pct_net` | float | `3PT%` minus `3PT%D` — how much better your 3PT% is than what you allow |
| `three_rate_net` | float | `3PTR` minus `3PTRD` — whether you attempt more 3s than opponents do against you |
| `three_value` | float | 3PT scoring contribution proxy: `3PT%` x 3 x (`3PTR` / 100) |
| `3PA` | int | Back-calculated season 3-point attempts |
| `3PM` | int | Back-calculated season 3-point makes |
| `2PA` | int | Back-calculated season 2-point attempts |
| `2PM` | int | Back-calculated season 2-point makes |
| `has_shot_splits` | int | Binary — 1 if Shooting Splits data available (2010+), else 0 |

In [ ]:
# Programmatic column summary: dtype, null count, unique count, sample value
summary = pd.DataFrame({
    'dtype'   : df.dtypes.astype(str),
    'non_null': df.count(),
    'null'    : df.isna().sum(),
    'null_%'  : (df.isna().mean() * 100).round(1),
    'unique'  : df.nunique(),
    'sample'  : [df[c].dropna().iloc[0] if df[c].count() > 0 else None for c in df.columns]
})
summary

In [ ]:
df.head(5)

---
# Descriptive Analysis

## 1. Summary Statistics

Central tendency and spread for the key 3PT and performance columns.

### Central Tendency — Mean, Median, Mode

In [ ]:
key_cols = [
    '3PT%', '3PTR', '3PT%D', '3PTRD',
    'three_pct_net', 'three_rate_net', 'three_value',
    'win_pct', 'KADJ EM', 'BARTHAG', 'EFG%', 'EFG%D'
]

central = pd.DataFrame({
    'Mean'  : df[key_cols].mean().round(3),
    'Median': df[key_cols].median().round(3),
    'Mode'  : df[key_cols].mode().iloc[0].round(3),
})

print('Central Tendency — Key Columns')
central

### Spread — Range, Standard Deviation, Variance, IQR

In [ ]:
spread = pd.DataFrame({
    'Min'     : df[key_cols].min().round(3),
    'Max'     : df[key_cols].max().round(3),
    'Range'   : (df[key_cols].max() - df[key_cols].min()).round(3),
    'Std Dev' : df[key_cols].std().round(3),
    'Variance': df[key_cols].var().round(3),
    'IQR'     : (df[key_cols].quantile(0.75) - df[key_cols].quantile(0.25)).round(3),
})

print('Spread — Key Columns')
spread

In [ ]:
# Summary statistics split by tournament status
print('=== Mean values: Tournament vs Non-Tournament ===')
print(df.groupby('made_tournament')[key_cols].mean().round(3).T.rename(
    columns={0: 'Non-Tournament', 1: 'Tournament'}
).to_string())

## 2. Aggregation

### Categorical Aggregation

Group and summarise 3PT stats by tournament stage, conference, and 3PT attempt rate quartile.

In [ ]:
# By tournament stage
round_labels = {
    0:'No Tourney', 1:'First Four', 2:'R64',
    3:'R32', 4:'S16', 5:'E8', 6:'F4', 7:'Final', 8:'Champion'
}
stage_order = list(round_labels.values())

stage_agg = (
    df.copy()
    .assign(Stage=df['tournament_round'].map(round_labels))
    .groupby('Stage')[['3PT%', '3PTR', '3PT%D', '3PTRD', 'win_pct', 'KADJ EM']]
    .agg(['mean', 'median', 'std'])
    .round(2)
    .reindex(stage_order)
)

print('Aggregation by Tournament Stage:')
stage_agg

In [ ]:
# By 3PT attempt rate quartile — does shooting more 3s correlate with better outcomes?
rate_agg = (
    df.groupby('three_rate_category', observed=True)
    [['3PT%', 'win_pct', 'KADJ EM', 'BARTHAG', 'made_tournament', 'reached_s16']]
    .agg(['mean', 'median', 'count'])
    .round(3)
)

print('Aggregation by 3PT Attempt Rate Quartile:')
rate_agg

In [ ]:
# By conference — top 12 conferences by team count
top_confs = df['CONF'].value_counts().head(12).index

conf_agg = (
    df[df['CONF'].isin(top_confs)]
    .groupby('CONF')[['3PT%', '3PTR', 'win_pct', 'KADJ EM', 'made_tournament']]
    .mean()
    .round(3)
    .sort_values('KADJ EM', ascending=False)
)

print('Aggregation by Conference (top 12 by team count):')
conf_agg

### Rolling Aggregation

3-year rolling averages of league-wide 3PT stats to smooth year-to-year noise and reveal longer-term trends.

In [ ]:
# League-wide yearly averages
yearly_avg = df.groupby('YEAR')[['3PT%', '3PTR', '3PT%D', '3PTRD',
                                   'EFG%', 'win_pct', 'KADJ EM']].mean().round(3)

# 3-year rolling mean
rolling_3yr = yearly_avg.rolling(window=3, min_periods=1).mean().round(3)
rolling_3yr.columns = [f'{c} (3yr roll)' for c in rolling_3yr.columns]

roll_display = pd.concat([yearly_avg[['3PT%', '3PTR', '3PT%D', '3PTRD']],
                           rolling_3yr[['3PT% (3yr roll)', '3PTR (3yr roll)']]], axis=1)

print('Yearly averages + 3-year rolling mean:')
roll_display

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, title in zip(axes,
    ['3PT%', '3PTR'],
    ['League-Wide 3PT FG% Over Time', 'League-Wide 3PT Attempt Rate Over Time']):

    ax.plot(yearly_avg.index, yearly_avg[col],
            marker='o', color='steelblue', linewidth=1.8, alpha=0.7, label='Yearly avg')
    ax.plot(rolling_3yr.index, rolling_3yr[f'{col} (3yr roll)'],
            color='crimson', linewidth=2.2, linestyle='--', label='3-yr rolling avg')
    ax.set_xlabel('Year')
    ax.set_ylabel(col)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting Trends — All D1 Teams', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Visualisation Techniques

### Charts and Graphs

In [ ]:
# Bar chart: avg 3PT% and 3PTR by tournament stage
stage_means = (
    df.copy()
    .assign(Stage=df['tournament_round'].map(round_labels))
    .groupby('Stage', observed=True)[['3PT%', '3PTR']]
    .mean()
    .reindex(stage_order)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.viridis([i / len(stage_order) for i in range(len(stage_order))])

for ax, col, ylabel in zip(axes, ['3PT%', '3PTR'], ['3PT FG%', '3PT Attempt Rate (%)']):
    bars = ax.bar(stage_means.index, stage_means[col], color=colors, edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Tournament Stage')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Avg {ylabel} by Tournament Stage', fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    ax.bar_label(bars, fmt='%.1f', fontsize=8, padding=2)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting by Tournament Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: 3PT stats vs win_pct and KADJ EM
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

pairs = [
    ('3PTR',           'win_pct',  '3PT Attempt Rate', 'Win %'),
    ('3PT%',           'win_pct',  '3PT FG%',          'Win %'),
    ('three_pct_net',  'KADJ EM',  '3PT% Net (Off-Def)', 'Adj. Efficiency Margin'),
    ('three_value',    'win_pct',  '3PT Value Score',  'Win %'),
]

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes, pairs):
    data = df[[xcol, ycol]].dropna()
    ax.scatter(data[xcol], data[ycol], alpha=0.25, s=10, color='steelblue')
    slope, intercept, r, p, _ = stats.linregress(data[xcol], data[ycol])
    x_line = [data[xcol].min(), data[xcol].max()]
    y_line = [slope * x + intercept for x in x_line]
    ax.plot(x_line, y_line, color='crimson', linewidth=2,
            label=f'r = {r:.2f}  p = {p:.3f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{xlabel} vs {ylabel}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.suptitle('3PT Shooting vs Performance — Scatter Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Histograms and Box Plots

In [ ]:
# Histograms: distribution of 3PT stats split by tournament status
hist_cols = [
    ('3PT%',  '3PT FG%'),
    ('3PTR',  '3PT Attempt Rate'),
    ('3PT%D', 'Opp 3PT FG% Allowed'),
    ('3PTRD', 'Opp 3PT Attempt Rate'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

palette = {0: ('#d9534f', 'Non-Tournament'), 1: ('#5cb85c', 'Tournament')}

for ax, (col, label) in zip(axes, hist_cols):
    for status, (color, name) in palette.items():
        data = df[df['made_tournament'] == status][col].dropna()
        ax.hist(data, bins=35, color=color, alpha=0.55, edgecolor='white',
                linewidth=0.3, label=f'{name} (n={len(data):,})', density=True)
        ax.axvline(data.mean(), color=color, linewidth=1.8, linestyle='--')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {label}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting Distributions — Tournament vs Non-Tournament', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: 3PT stats across tournament stages
box_cols = [
    ('3PT%',  '3PT FG%'),
    ('3PTR',  '3PT Attempt Rate'),
    ('three_pct_net', '3PT% Net (Off - Def)'),
    ('win_pct', 'Win %'),
]

plot_df = df.copy()
plot_df['Stage'] = pd.Categorical(
    df['tournament_round'].map(round_labels),
    categories=stage_order, ordered=True
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, (col, label) in zip(axes, box_cols):
    sns.boxplot(x='Stage', y=col, data=plot_df, order=stage_order,
                palette='viridis', ax=ax, linewidth=0.8,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
    means = plot_df.groupby('Stage', observed=True)[col].mean().reindex(stage_order)
    ax.plot(range(len(stage_order)), means.values, 'D--',
            color='white', markersize=5, markeredgecolor='black',
            linewidth=1.5, label='Mean')
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Tournament Stage', fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting & Win Rate Across Tournament Stages', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Time Series Analysis

Using year as the time axis (2008–2026). Each data point is the league-wide average across all D1 teams for that season.

> **Note:** 2020 is absent — the NCAA Tournament was cancelled due to COVID-19.

### Trend

Long-run direction of 3PT shooting over time.

In [ ]:
from scipy import stats as scipy_stats

ts_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'EFG%', 'win_pct']
ts = df.groupby('YEAR')[ts_cols].mean()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, ts_cols):
    years = ts.index.values
    values = ts[col].values

    # OLS trend line
    slope, intercept, r, p, _ = scipy_stats.linregress(years, values)
    trend_line = slope * years + intercept

    ax.plot(years, values, marker='o', color='steelblue',
            linewidth=2, markersize=5, label='Yearly avg')
    ax.plot(years, trend_line, color='crimson', linewidth=2,
            linestyle='--', label=f'Trend  r={r:.2f}  slope={slope:+.3f}/yr')
    ax.set_xlabel('Year')
    ax.set_ylabel(col)
    ax.set_title(f'{col} — Trend', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Long-Run Trends in 3PT Shooting & Performance (2008–2026)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Seasonality

Cyclic patterns in the time series — deviations from the overall trend that repeat over a multi-year window.
Using a 4-year rolling window to capture one full recruiting cycle.

In [ ]:
# Detrend: subtract OLS trend, then show the cyclical residual
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, ['3PT%', '3PTR']):
    years  = ts.index.values.astype(float)
    values = ts[col].values

    # OLS trend
    slope, intercept, *_ = scipy_stats.linregress(years, values)
    trend   = slope * years + intercept
    detrend = values - trend          # remove trend to expose cycle

    # 4-year centred rolling mean of the detrended series (cyclical component)
    detrend_s = pd.Series(detrend, index=ts.index)
    cycle = detrend_s.rolling(window=4, center=True, min_periods=2).mean()

    ax.bar(ts.index, detrend, color='steelblue', alpha=0.5,
           label='Detrended (yearly)', width=0.6)
    ax.plot(ts.index, cycle, color='crimson', linewidth=2.5,
            label='Cyclical (4-yr centred roll)')
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'{col} deviation from trend')
    ax.set_title(f'{col} — Cyclical Component', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Seasonality / Cyclical Patterns in 3PT Shooting (detrended)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Residual Component

What is left after removing the trend and cyclical components — year-specific noise or shocks (e.g. rule changes, COVID).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, ['3PT%', '3PTR']):
    years  = ts.index.values.astype(float)
    values = ts[col].values

    # Trend
    slope, intercept, *_ = scipy_stats.linregress(years, values)
    trend   = slope * years + intercept
    detrend = values - trend

    detrend_s = pd.Series(detrend, index=ts.index)
    cycle     = detrend_s.rolling(window=4, center=True, min_periods=2).mean()
    residual  = detrend_s - cycle

    ax.bar(ts.index, residual, color='darkorange', alpha=0.7, width=0.6,
           label='Residual')
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')

    # Annotate the largest shocks
    for yr in residual.abs().nlargest(3).index:
        ax.annotate(str(yr), (yr, residual[yr]),
                    textcoords='offset points',
                    xytext=(0, 8 if residual[yr] >= 0 else -14),
                    ha='center', fontsize=8, color='black', fontweight='bold')

    ax.set_xlabel('Year')
    ax.set_ylabel(f'{col} residual')
    ax.set_title(f'{col} — Residual Component', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Residuals after Trend + Cycle Removal — 3PT Shooting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Full decomposition table: Actual = Trend + Cycle + Residual
decomp_records = []

for col in ['3PT%', '3PTR']:
    years  = ts.index.values.astype(float)
    values = ts[col].values
    slope, intercept, *_ = scipy_stats.linregress(years, values)
    trend     = pd.Series(slope * years + intercept, index=ts.index)
    detrend_s = pd.Series(values, index=ts.index) - trend
    cycle     = detrend_s.rolling(window=4, center=True, min_periods=2).mean()
    residual  = detrend_s - cycle

    for yr in ts.index:
        decomp_records.append({
            'YEAR': yr, 'Column': col,
            'Actual'  : round(ts.loc[yr, col], 3),
            'Trend'   : round(trend[yr], 3),
            'Cycle'   : round(cycle[yr], 3) if not pd.isna(cycle[yr]) else None,
            'Residual': round(residual[yr], 3) if not pd.isna(residual[yr]) else None,
        })

decomp_df = pd.DataFrame(decomp_records)
print('Decomposition table (Actual = Trend + Cycle + Residual):')
decomp_df

---
# Diagnostic Analysis

**Design Thinking Question:** Does 3-point shooting affect win rate in NCAA basketball?

Diagnostic analysis moves beyond *what* happened to *why* — starting from a specific observation in the data, then systematically investigating its cause.

## Observation

Before running any diagnostic tests, we first identify a concrete pattern from the descriptive analysis to investigate.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# Compute the key descriptive numbers that lead to the observation
tourney     = df[df['made_tournament'] == 1]
non_tourney = df[df['made_tournament'] == 0]

obs_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'win_pct', 'KADJ EM']
obs_summary = pd.DataFrame({
    'Non-Tournament mean': non_tourney[obs_cols].mean().round(3),
    'Tournament mean'    : tourney[obs_cols].mean().round(3),
    'Difference'         : (tourney[obs_cols].mean() - non_tourney[obs_cols].mean()).round(3),
})
print('Mean comparison — Tournament vs Non-Tournament teams:')
obs_summary

In [ ]:
# Visualise the observation: 3PT% vs 3PTR relationship with win rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {0: ('#d9534f', 'Non-Tournament'), 1: ('#5cb85c', 'Tournament')}

for ax, (xcol, xlabel) in zip(axes, [('3PT%', '3PT FG%'), ('3PTR', '3PT Attempt Rate')]):
    for status, (color, label) in palette.items():
        sub = df[df['made_tournament'] == status]
        ax.scatter(sub[xcol], sub['win_pct'], color=color, alpha=0.25, s=10, label=label)
        slope, intercept, r, p, _ = scipy_stats.linregress(
            sub[xcol].dropna(), sub.loc[sub[xcol].notna(), 'win_pct'])
        x_line = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=2,
                label=f'{label}  r={r:.2f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Win %')
    ax.set_title(f'{xlabel} vs Win Rate', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Observation: 3PT Efficiency vs Volume — Which Drives Win Rate?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Observation Statement

> **3-point shooting efficiency (3PT%) is positively correlated with win rate — but 3-point attempt rate (3PTR) shows little to no relationship with win rate.**
>
> In other words: shooting 3-pointers *accurately* separates winners from losers, but shooting *more* 3-pointers does not.

This is counter-intuitive. If 3-pointers are worth more points, why doesn't attempting more of them lead to more wins?

The following diagnostic analysis investigates *why* this pattern exists.

## 1. Relationship and Dependency Analysis

Quantify and map the relationships between 3PT shooting variables and win rate to understand the structure of the observation.

### Correlation Analysis

**Pearson** measures the linear relationship; **Spearman** measures the monotonic relationship (rank-based, more robust to outliers).

Comparing both tells us whether the 3PT → win rate link is linear or non-linear.

In [ ]:
target_cols  = ['win_pct', 'KADJ EM', 'BARTHAG', 'tournament_round']
feature_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD',
                'three_pct_net', 'three_rate_net', 'three_value',
                'EFG%', 'EFG%D', 'TALENT', 'WAB']

clean = df[feature_cols + target_cols].dropna()

pearson_r  = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)
pearson_p  = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)
spearman_r = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)
spearman_p = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)

for feat in feature_cols:
    for tgt in target_cols:
        pr, pp = scipy_stats.pearsonr(clean[feat], clean[tgt])
        sr, sp = scipy_stats.spearmanr(clean[feat], clean[tgt])
        pearson_r.loc[feat, tgt]  = round(pr, 3)
        pearson_p.loc[feat, tgt]  = round(pp, 4)
        spearman_r.loc[feat, tgt] = round(sr, 3)
        spearman_p.loc[feat, tgt] = round(sp, 4)

# Side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, mat, pmat, title in zip(axes,
    [pearson_r, spearman_r], [pearson_p, spearman_p],
    ['Pearson r', 'Spearman r']):

    sig_mask = pmat > 0.05
    sns.heatmap(mat.astype(float), annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, linewidths=0.4, linecolor='white',
                annot_kws={'size': 9}, ax=ax,
                mask=sig_mask, cbar_kws={'label': title})
    sns.heatmap(mat.astype(float), annot=False, cmap='Greys',
                center=0, alpha=0.25, linewidths=0.4, linecolor='white',
                ax=ax, mask=~sig_mask, cbar=False)
    ax.set_title(f'{title} — Features vs Outcomes\n(greyed = p > 0.05)',
                 fontweight='bold', fontsize=11)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.suptitle('Correlation: Does 3PT shooting drive win rate?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare Pearson vs Spearman for win_pct — focus column
compare = pd.DataFrame({
    'Pearson r'   : pearson_r['win_pct'],
    'Pearson p'   : pearson_p['win_pct'],
    'Spearman r'  : spearman_r['win_pct'],
    'Spearman p'  : spearman_p['win_pct'],
    'Sig (p<0.05)': (pearson_p['win_pct'] < 0.05).map({True: 'Yes', False: 'No'})
}).sort_values('Pearson r', key=abs, ascending=False)

print('Pearson vs Spearman — correlation with win_pct:')
print()
print('Key finding: 3PT% r =', pearson_r.loc['3PT%', 'win_pct'],
      '| 3PTR r =', pearson_r.loc['3PTR', 'win_pct'])
print('This confirms the observation: efficiency correlates with wins; volume does not.')
compare

### Regression Analysis

Estimates the size and shape of the 3PT → win rate relationship.

- **Linear Regression** — how much does win rate change per 1-unit increase in 3PT%?
- **Polynomial Regression** — is the relationship curved? Do diminishing returns exist at high 3PT%?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

reg_data = df[['3PT%', '3PTR', '3PT%D', '3PTRD',
               'three_pct_net', 'EFG%', 'EFG%D', 'TALENT', 'win_pct', 'KADJ EM']].dropna()

y = reg_data['win_pct']
print(f'Regression dataset: {len(reg_data):,} rows')

In [ ]:
# Simple linear regression: each 3PT feature vs win_pct
features = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net', 'EFG%', 'EFG%D']
results  = []

for feat in features:
    x   = reg_data[[feat]]
    lr  = LinearRegression().fit(x, y)
    yp  = lr.predict(x)
    r2  = r2_score(y, yp)
    rmse = np.sqrt(mean_squared_error(y, yp))
    results.append({
        'Feature'  : feat,
        'Coeff'    : round(lr.coef_[0], 4),
        'Intercept': round(lr.intercept_, 3),
        'R²'       : round(r2, 4),
        'RMSE'     : round(rmse, 4),
        'Interpet' : f'+{lr.coef_[0]:.3f}% win rate per 1-unit increase in {feat}'
                      if lr.coef_[0] > 0
                      else f'{lr.coef_[0]:.3f}% win rate per 1-unit increase in {feat}'
    })

simple_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print('Simple Linear Regression vs win_pct:')
simple_df

In [ ]:
# Multiple regression: 3PT-only vs full model
X_3pt = reg_data[['3PT%', '3PTR', '3PT%D', '3PTRD']]
X_full = reg_data[['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net', 'EFG%', 'EFG%D', 'TALENT']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, X) in zip(axes, [('3PT-Only Model', X_3pt), ('Full Model', X_full)]):
    lr    = LinearRegression().fit(X, y)
    y_pred = lr.predict(X)
    cv_r2 = cross_val_score(lr, X, y, cv=5, scoring='r2').mean()
    r2    = r2_score(y, y_pred)

    ax.scatter(y, y_pred, alpha=0.2, s=10, color='steelblue')
    lim = [y.min(), y.max()]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Win %')
    ax.set_ylabel('Predicted Win %')
    ax.set_title(f'{label}\nR² = {r2:.3f}  |  CV R² = {cv_r2:.3f}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Actual vs Predicted Win Rate — Linear Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Coefficient comparison
lr_3pt  = LinearRegression().fit(X_3pt, y)
lr_full = LinearRegression().fit(X_full, y)
coef_df = pd.DataFrame({
    '3PT-Only coeff': pd.Series(dict(zip(X_3pt.columns, lr_3pt.coef_.round(4)))),
    'Full coeff'    : pd.Series(dict(zip(X_full.columns, lr_full.coef_.round(4)))),
})
print('Coefficients:')
coef_df

In [ ]:
# Polynomial regression: 3PT% vs win_pct — does the curve fit better?
x_raw   = reg_data[['3PT%']].values
y_vals  = y.values
x_range = np.linspace(x_raw.min(), x_raw.max(), 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, degree in zip(axes, [1, 2, 3]):
    poly     = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly   = poly.fit_transform(x_raw)
    lr_poly  = LinearRegression().fit(X_poly, y_vals)
    r2_train = r2_score(y_vals, lr_poly.predict(X_poly))
    cv_r2    = cross_val_score(lr_poly, X_poly, y_vals, cv=5, scoring='r2').mean()
    y_curve  = lr_poly.predict(poly.transform(x_range))

    ax.scatter(x_raw, y_vals, alpha=0.2, s=8, color='steelblue', label='Data')
    ax.plot(x_range, y_curve, color='crimson', linewidth=2.2,
            label=f'Degree {degree}  R²={r2_train:.3f}  CV={cv_r2:.3f}')
    ax.set_xlabel('3PT%')
    ax.set_ylabel('Win %')
    ax.set_title(f'Polynomial Degree {degree}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Polynomial Regression — 3PT% vs Win Rate\n(CV R² tells us if a curve actually fits better out-of-sample)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Root Cause Identification

**Observation to explain:** 3PT% correlates with win rate but 3PTR does not.

Systematic process: identify the problem → map possible causes → find the root → propose corrective actions → define monitoring.

### Step 1 — Identify the Problem (Scope, Impact, 5W)

| | |
|---|---|
| **Who** | All NCAA D1 teams (2008–2026), ~350 teams per season |
| **What** | 3PT% correlates with win rate (r ≈ +0.3); 3PTR does not (r ≈ 0). Teams shooting more 3s are not winning more games |
| **Where** | NCAA college basketball — all conferences, all tournament stages |
| **When** | Consistent across all 17 seasons in the dataset (2008–2026) |
| **Why** | Understanding this separates *style* (how many 3s you take) from *skill* (how accurately you shoot them) — critical for team strategy and recruitment |

**Scope:** The gap in 3PT% between tournament and non-tournament teams is small (~0.5–1%), but the win rate gap is large (~25%). This suggests 3PT% is one of many correlated factors, not the sole driver.

**Impact:** If teams over-invest in 3PT volume without improving accuracy, they may see no win rate benefit.

In [ ]:
# Quantify the scope
print('=== Win Rate Gap ===')
print(f'Tournament teams mean win_pct     : {tourney["win_pct"].mean():.1f}%')
print(f'Non-tournament teams mean win_pct : {non_tourney["win_pct"].mean():.1f}%')
print(f'Gap                               : {tourney["win_pct"].mean() - non_tourney["win_pct"].mean():.1f}%')
print()
print('=== 3PT Gap ===')
print(f'Tournament teams mean 3PT%        : {tourney["3PT%"].mean():.2f}%')
print(f'Non-tournament teams mean 3PT%    : {non_tourney["3PT%"].mean():.2f}%')
print(f'Gap                               : {tourney["3PT%"].mean() - non_tourney["3PT%"].mean():.2f}%')
print()
print(f'Tournament teams mean 3PTR        : {tourney["3PTR"].mean():.2f}%')
print(f'Non-tournament teams mean 3PTR    : {non_tourney["3PTR"].mean():.2f}%')
print(f'Gap                               : {tourney["3PTR"].mean() - non_tourney["3PTR"].mean():.2f}%')
print()

# % of win rate variance explained by 3PT% alone vs 3PTR alone
tmp = df[['3PT%', '3PTR', 'win_pct']].dropna()
r2_pct = r2_score(tmp['win_pct'], LinearRegression().fit(tmp[['3PT%']], tmp['win_pct']).predict(tmp[['3PT%']]))
r2_ptr = r2_score(tmp['win_pct'], LinearRegression().fit(tmp[['3PTR']], tmp['win_pct']).predict(tmp[['3PTR']]))
print(f'R² of win_pct ~ 3PT%  : {r2_pct:.4f}  ({r2_pct*100:.2f}%)')
print(f'R² of win_pct ~ 3PTR  : {r2_ptr:.4f}  ({r2_ptr*100:.2f}%)')

### Step 2 — Possible Causal Reasons (Fishbone Diagram)

Mapping all plausible reasons why 3PT% correlates with wins but 3PTR does not.

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(20, 10))
ax.set_xlim(0, 20); ax.set_ylim(0, 10)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# Spine
ax.annotate('', xy=(18, 5), xytext=(1, 5),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=3.5))

# Effect box
ebox = mpatches.FancyBboxPatch((18.05, 4.15), 1.7, 1.7,
    boxstyle='round,pad=0.15', fc='#e74c3c', ec='white', lw=2)
ax.add_patch(ebox)
ax.text(18.9, 5, '3PT%\ncorrelates\nwith wins\n(3PTR does not)',
        ha='center', va='center', fontsize=8, fontweight='bold', color='white')

branches = [
    # (label, x_spine, y_dir, colour, sub-causes)
    ('Shooting Skill\n& Selection', 3.5, +1, '#2980b9', [
        'Accurate shooters raise 3PT%',
        'Good shooters take fewer forced 3s',
        'Shot quality > shot quantity',
        'Open 3s go in at higher rate',
    ]),
    ('Team Efficiency\nEffect', 7.5, +1, '#8e44ad', [
        '3PT% is part of EFG%',
        'EFG% drives KADJ EM',
        'KADJ EM strongly predicts wins',
        '3PT% is a symptom of efficiency',
    ]),
    ('Opponent\nDefence', 13.0, +1, '#27ae60', [
        'Good defences suppress 3PT%',
        'Bad teams face weaker defences',
        '3PTRD (opp attempt rate) confounds',
        'Volume ≠ quality of attempts',
    ]),
    ('Talent &\nRecruitment', 4.5, -1, '#e67e22', [
        'Talented shooters raise 3PT%',
        'Better teams recruit better shooters',
        'Talent correlates with all outcomes',
        '3PTR not a talent signal',
    ]),
    ('Strategy &\nCoaching', 9.5, -1, '#c0392b', [
        'Some teams force high-volume 3s',
        'High 3PTR teams may sacrifice 2PT',
        'Coaching selects for accuracy',
        'System fit raises 3PT%',
    ]),
    ('Variance &\nConfounding', 14.5, -1, '#16a085', [
        '3PTR is highly variable year-to-year',
        'Small 3PT% edge compounds over games',
        'Win rate averages out volume noise',
        'EFG%, KADJ EM mask 3PTR signal',
    ]),
]

for label, x_root, y_dir, color, sub_causes in branches:
    y_root = 5
    y_tip  = 5 + y_dir * 2.5
    x_tip  = x_root - 0.5

    ax.plot([x_tip, x_root], [y_tip, y_root], color=color, lw=2.8, solid_capstyle='round')
    ax.text(x_tip, y_tip + y_dir * 0.4, label, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white',
            bbox=dict(fc=color, ec='white', boxstyle='round,pad=0.3', lw=1.5))

    n = len(sub_causes)
    for i, cause in enumerate(sub_causes):
        t  = 0.2 + i * (0.6 / max(n - 1, 1))
        xr = x_tip + t * (x_root - x_tip)
        yr = y_tip + t * (y_root - y_tip)
        xs = xr - 0.4
        ys = yr + y_dir * 0.85
        ax.plot([xs, xr], [ys, yr], color=color, lw=1.3, alpha=0.9)
        ax.text(xs, ys + y_dir * 0.2, cause, ha='center', va='center',
                fontsize=6.8, color='#2c3e50',
                bbox=dict(fc='white', ec=color, alpha=0.85, boxstyle='round,pad=0.12', lw=0.7))

ax.set_title('Fishbone Diagram — Why does 3PT% predict wins but 3PTR does not?',
             fontsize=14, fontweight='bold', pad=15, color='#2c3e50')
plt.tight_layout()
plt.show()

### Step 3 — Determine Root Causes

Rank every candidate explanation by how much it explains the observation using individual R².

In [ ]:
candidates = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net',
              'EFG%', 'EFG%D', 'KADJ O', 'KADJ D', 'TALENT', 'EXP', 'WAB']

root_data = df[candidates + ['win_pct']].dropna()
root_rows = []

for feat in candidates:
    lr  = LinearRegression().fit(root_data[[feat]], root_data['win_pct'])
    r2  = r2_score(root_data['win_pct'], lr.predict(root_data[[feat]]))
    r, p = scipy_stats.pearsonr(root_data[feat], root_data['win_pct'])
    cat = ('3PT Shooting' if '3PT' in feat or 'three' in feat
           else 'Overall Efficiency' if feat in ['EFG%', 'EFG%D', 'KADJ O', 'KADJ D']
           else 'Team Quality')
    root_rows.append({'Feature': feat, 'Pearson r': round(r,3), 'R²': round(r2,4),
                       'p-value': round(p,4), 'Category': cat})

root_df = pd.DataFrame(root_rows).sort_values('R²', ascending=False)

cat_colors = {'3PT Shooting': '#2980b9', 'Overall Efficiency': '#27ae60', 'Team Quality': '#e67e22'}

fig, ax = plt.subplots(figsize=(11, 6))
bar_colors = [cat_colors[r['Category']] for _, r in root_df.iterrows()]
bars = ax.barh(root_df['Feature'], root_df['R²'], color=bar_colors, edgecolor='white')
ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=3)
ax.set_xlabel('R² with Win Rate')
ax.set_title('Root Cause Ranking — What actually drives win rate?\n(Confirms 3PT% matters less than overall efficiency)',
             fontweight='bold')
ax.grid(axis='x', alpha=0.3)
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
ax.legend(handles=legend_patches, fontsize=9)
plt.tight_layout()
plt.show()

print('Root cause table:')
root_df

### Step 4 — Corrective Actions

Based on the root causes, what should teams actually do?

In [ ]:
# Show win rate outcomes by 3PT% quartile and 3PTR quartile separately
plot_df = df.dropna(subset=['three_rate_category', 'three_pct_category', 'win_pct'])
cat_order = ['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cat_col, title, action in zip(axes,
    ['three_pct_category', 'three_rate_category'],
    ['Win Rate by 3PT FG% Quartile', 'Win Rate by 3PT Attempt Rate Quartile'],
    ['Improve shooting accuracy → win more', 'Shoot more 3s → no clear benefit']):

    grp = plot_df.groupby(cat_col, observed=True)['win_pct'].agg(['mean','median','std']).reindex(cat_order)
    x = range(len(cat_order))
    ax.bar(x, grp['mean'], color='steelblue', alpha=0.75, label='Mean win %', width=0.45)
    ax.bar([i + 0.47 for i in x], grp['median'], color='darkorange', alpha=0.75, label='Median win %', width=0.45)
    ax.errorbar(x, grp['mean'], yerr=grp['std'], fmt='none', color='black', capsize=4, lw=1.2)
    ax.set_xticks([i + 0.235 for i in x])
    ax.set_xticklabels(cat_order, rotation=15)
    ax.set_ylabel('Win %')
    ax.set_title(f'{title}\n→ {action}', fontweight='bold', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Corrective Action View — Accuracy matters; volume alone does not',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 5 — Follow-Up (Monitoring)

Track 3PT% and 3PTR year-over-year to monitor whether the observation holds or shifts as the game evolves.

In [ ]:
yearly = df.groupby('YEAR')[['3PT%', '3PTR', 'win_pct', 'KADJ EM']].mean()
yearly_delta = yearly.diff().round(3)
yearly_delta.columns = [f'Δ {c}' for c in yearly_delta.columns]

# Check if 3PT% ↑ years are also win_pct ↑ years (should align if causal)
monitor = pd.concat([yearly.round(3), yearly_delta], axis=1)
corr_pct_win = monitor['3PT%'].corr(monitor['win_pct'])
corr_ptr_win = monitor['3PTR'].corr(monitor['win_pct'])

print(f'Year-level correlation: Δ3PT% vs Δwin_pct : {yearly["3PT%"].corr(yearly["win_pct"]):.3f}')
print(f'Year-level correlation: Δ3PTR vs Δwin_pct : {yearly["3PTR"].corr(yearly["win_pct"]):.3f}')
print()
print('Annual monitoring table:')
monitor

## 3. Hypothesis Testing and Validation

Formally test whether the observation is statistically significant or could be due to chance.

**Observation being tested:** 3PT% differs between tournament and non-tournament teams; 3PTR does not.

All tests use **α = 0.05**.

### t-Test — Does 3PT% differ significantly between tournament and non-tournament teams?

| | |
|---|---|
| **H₀** | Tournament and non-tournament teams have the same mean 3PT% |
| **H₁** | Tournament teams have a significantly higher mean 3PT% |
| **Test** | Welch's t-test (does not assume equal variances) |

In [ ]:
from scipy.stats import ttest_ind

t_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net', 'win_pct', 'KADJ EM']
t_rows = []

for col in t_cols:
    g1 = tourney[col].dropna()
    g0 = non_tourney[col].dropna()
    t_stat, p_val = ttest_ind(g1, g0, equal_var=False)
    t_rows.append({
        'Column'              : col,
        'Tournament mean'     : round(g1.mean(), 3),
        'Non-Tournament mean' : round(g0.mean(), 3),
        'Difference'          : round(g1.mean() - g0.mean(), 3),
        't-statistic'         : round(t_stat, 3),
        'p-value'             : round(p_val, 4),
        'Significant (p<0.05)': 'Yes' if p_val < 0.05 else 'No',
    })

t_df = pd.DataFrame(t_rows)
print('Welch t-Test — Tournament vs Non-Tournament:')
t_df

### Chi-Square Test — Is 3PT attempt rate quartile associated with tournament qualification?

| | |
|---|---|
| **H₀** | 3PT attempt rate quartile and tournament qualification are independent |
| **H₁** | There is a significant association between 3PT attempt rate and making the tournament |
| **Expected result** | If 3PTR truly doesn't matter, H₀ should not be rejected |

In [ ]:
from scipy.stats import chi2_contingency

chi_data    = df.dropna(subset=['three_rate_category', 'made_tournament'])
contingency = pd.crosstab(chi_data['three_rate_category'], chi_data['made_tournament'],
                           rownames=['3PT Rate Category'], colnames=['Made Tournament'])

chi2, p, dof, expected = chi2_contingency(contingency)

print('Contingency table (observed):')
print(contingency)
print()
print(f'Chi-Square statistic : {chi2:.3f}')
print(f'Degrees of freedom   : {dof}')
print(f'p-value              : {p:.4f}')
print(f'Significant (p<0.05) : {"Yes — 3PTR quartile is associated with tournament qualification" if p < 0.05 else "No — 3PTR quartile is independent of tournament qualification"}')
print()
print('Expected frequencies:')
print(pd.DataFrame(expected.round(1), index=contingency.index, columns=contingency.columns))

### ANOVA — Does 3PT% differ significantly across all tournament stages?

| | |
|---|---|
| **H₀** | Mean 3PT% is the same across all tournament stages (No Tourney → Champion) |
| **H₁** | At least one stage has a significantly different mean 3PT% |
| **Tests** | One-way ANOVA (parametric) + Kruskal-Wallis (non-parametric, no normality assumption) |

In [ ]:
from scipy.stats import f_oneway, kruskal

round_labels = {0:'No Tourney', 1:'First Four', 2:'R64', 3:'R32',
                4:'S16', 5:'E8', 6:'F4', 7:'Final', 8:'Champion'}

anova_df         = df.copy()
anova_df['Stage'] = anova_df['tournament_round'].map(round_labels)
stage_order_list  = list(round_labels.values())

anova_rows = []
for col in ['3PT%', '3PTR', 'three_pct_net']:
    groups = [anova_df[anova_df['Stage'] == s][col].dropna().values
              for s in stage_order_list]

    f_stat, f_p   = f_oneway(*groups)
    h_stat, h_p   = kruskal(*[g for g in groups if len(g) > 0])

    anova_rows += [
        {'Test': 'One-Way ANOVA',  'Metric': col, 'Statistic': round(f_stat,3), 'p-value': round(f_p,4), 'Significant': 'Yes' if f_p < 0.05 else 'No'},
        {'Test': 'Kruskal-Wallis', 'Metric': col, 'Statistic': round(h_stat,3), 'p-value': round(h_p,4), 'Significant': 'Yes' if h_p < 0.05 else 'No'},
    ]

anova_result = pd.DataFrame(anova_rows)
print('ANOVA / Kruskal-Wallis — 3PT stats across tournament stages:')
anova_result

### Validation Summary

Bringing the statistical results back to the observation.

In [ ]:
print('=' * 65)
print('DIAGNOSTIC ANALYSIS — SUMMARY')
print('=' * 65)
print()
print('OBSERVATION:')
print('  3PT% is positively correlated with win rate.')
print('  3PTR (attempt rate) shows little to no correlation.')
print()
print('RELATIONSHIP & DEPENDENCY:')
r_pct = pearson_r.loc['3PT%', 'win_pct']
r_ptr = pearson_r.loc['3PTR', 'win_pct']
print(f'  Pearson r  — 3PT% vs win_pct : {r_pct}')
print(f'  Pearson r  — 3PTR vs win_pct : {r_ptr}')
print(f'  3PT% explains {float(r_pct)**2*100:.1f}% of win rate variance; 3PTR explains {float(r_ptr)**2*100:.1f}%')
print()
print('ROOT CAUSE:')
print('  Shooting efficiency (3PT%) is a proxy for overall team quality.')
print('  EFG% and KADJ EM absorb most of its explanatory power.')
print('  High 3PTR without accuracy does not improve win rate.')
print()
print('HYPOTHESIS TESTS:')
for _, row in t_df[t_df['Column'].isin(['3PT%', '3PTR'])].iterrows():
    print(f'  t-test {row["Column"]:6s}: diff={row["Difference"]:+.3f}  p={row["p-value"]:.4f}  '
          f'Significant={row["Significant (p<0.05)"]}')
print(f'  Chi-sq 3PTR cat vs tourney: chi2={chi2:.3f}  p={p:.4f}  '
      f'Significant={"Yes" if p < 0.05 else "No"}')
print()
print('CONCLUSION:')
print('  Evidence SUPPORTS the observation.')
print('  3PT% differences are statistically significant (t-test).')
print('  However, 3PT% is partly a symptom of overall efficiency —')
print('  not the sole root cause of win rate variation.')